In [23]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


print("Libraries imported successfully!")


Libraries imported successfully!


In [24]:
def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    print(f"📉 Initial DataFrame Memory Usage: {start_mem:.2f} MB")
    
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and str(col_type) != 'category':
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                # SPECIAL CHECK: Time aur Amount ko float16 mein mat bhejo taaki overflow na ho!
                if col in ['Time', 'Amount']:
                    df[col] = df[col].astype(np.float32)
                else:
                    if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                        df[col] = df[col].astype(np.float16)
                    elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                        df[col] = df[col].astype(np.float32)
                    else:
                        df[col] = df[col].astype(np.float64)
        else:
            df[col] = df[col].astype('category')
            
    end_mem = df.memory_usage().sum() / 1024**2
    print(f"📈 Optimized Memory Usage: {end_mem:.2f} MB")
    print(f"✨ Memory Reduction: {100 * (start_mem - end_mem) / start_mem:.1f}%\n")
    return df

In [25]:
print("🚀 Loading dataset...")
df = pd.read_csv('creditcard.csv')


df = reduce_mem_usage(df)

🚀 Loading dataset...
📉 Initial DataFrame Memory Usage: 67.36 MB
📈 Optimized Memory Usage: 17.65 MB
✨ Memory Reduction: 73.8%



In [26]:
print(f"📊 Dataset Shape: {df.shape[0]:,} rows and {df.shape[1]} columns\n")

print("Checking for Missing Values:")
missing_vals = df.isnull().sum().sum()
print(f"Total Missing Values in Dataset: {missing_vals}")


df.head()

📊 Dataset Shape: 284,807 rows and 31 columns

Checking for Missing Values:
Total Missing Values in Dataset: 0


c:\Users\rites\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\io\formats\format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359375,-0.072754,2.537109,1.377930,-0.338379,0.462402,0.239624,0.098694,0.363770,...,-0.018311,0.277832,-0.110474,0.066956,0.128540,-0.189087,0.133545,-0.021057,149.619995,0
1,0.0,1.191406,0.266113,0.166504,0.448242,0.060028,-0.082336,-0.078796,0.085083,-0.255371,...,-0.225830,-0.638672,0.101318,-0.339844,0.167114,0.125854,-0.008980,0.014725,2.690000,0
2,1.0,-1.358398,-1.339844,1.773438,0.379883,-0.503418,1.800781,0.791504,0.247681,-1.514648,...,0.248047,0.771484,0.909180,-0.689453,-0.327637,-0.139038,-0.055359,-0.059753,378.660004,0
3,1.0,-0.966309,-0.185181,1.792969,-0.863281,-0.010307,1.247070,0.237549,0.377441,-1.386719,...,-0.108276,0.005272,-0.190308,-1.175781,0.647461,-0.221924,0.062744,0.061462,123.500000,0
4,2.0,-1.158203,0.877930,1.548828,0.403076,-0.407227,0.095947,0.592773,-0.270508,0.817871,...,-0.009430,0.798340,-0.137451,0.141235,-0.206055,0.502441,0.219482,0.215210,69.989998,0


In [27]:
print("📈 Statistical Summary of Amount & Time:")
df[['Time', 'Amount']].describe()

📈 Statistical Summary of Amount & Time:


,Time,Amount
count,284807.000000,284807.000000
mean,94813.859375,88.349617
std,47488.144531,250.120117
min,0.000000,0.000000
25%,54201.500000,5.600000
50%,84692.000000,22.000000
75%,139320.500000,77.165001
max,172792.000000,25691.160156


In [37]:
# Calculate class counts
fraud_counts = df['Class'].value_counts().reset_index()
fraud_counts.columns = ['Class', 'Count']
fraud_counts['Class_Label'] = fraud_counts['Class'].map({0: 'Legitimate (0)', 1: 'Fraud (1)'})

print("Class Distribution Breakdown:")
print(f"Legitimate Transactions (0): {fraud_counts.loc[fraud_counts['Class'] == 0, 'Count'].values[0]:,}")
print(f"Fraudulent Transactions (1): {fraud_counts.loc[fraud_counts['Class'] == 1, 'Count'].values[0]:,}")
fraud_pct = (fraud_counts.loc[fraud_counts['Class'] == 1, 'Count'].values[0] / len(df)) * 100
print(f"Fraud Percentage: {fraud_pct:.4f}%\n")

# Render Interactive Plotly Bar Chart directly inside Jupyter Notebook
fig = px.bar(
    fraud_counts,
    x='Class_Label',
    y='Count',
    title='<b>Class Imbalance Distribution (Legit vs Fraud)</b>',
    color='Class_Label',
    color_discrete_map={'Legitimate (0)': '#636EFA', 'Fraud (1)': '#EF553B'},
    text='Count'
)

fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(
    xaxis_title='Transaction Type',
    yaxis_title='Number of Transactions',
    template='plotly_dark',
    title_x=0.5
)
import plotly.io as pio
pio.renderers.default = 'browser'  # Ya tum 'browser' bhi likh sakte ho

# Ab apna graph show karo
fig.show()



Class Distribution Breakdown:
Legitimate Transactions (0): 284,315
Fraudulent Transactions (1): 492
Fraud Percentage: 0.1727%



In [42]:
# Cell for Amount Distribution
fig1 = px.histogram(
    df, x='Amount', color='Class',
    log_y=True, barmode='overlay',
    title='<b>Transaction Amount Distribution (Log Scale)</b>',
    color_discrete_map={0: '#636EFA', 1: '#EF553B'}
)
fig1.update_layout(template='plotly_dark', title_x=0.5, xaxis_title='Transaction Amount', yaxis_title='Log Count')
fig1.show(renderer='browser')

In [43]:
# Cell for Time & Hour Pattern Analysis
df['Hour'] = (df['Time'] / 3600) % 24

fig2 = px.histogram(
    df, x='Hour', color='Class',
    barmode='overlay', nbins=48,
    title='<b>Transaction Frequency by Hour of Day</b>',
    color_discrete_map={0: '#636EFA', 1: '#EF553B'}
)
fig2.update_layout(template='plotly_dark', title_x=0.5, xaxis_title='Hour of the Day (0-24)', yaxis_title='Transaction Count')
fig2.show(renderer='browser')

In [44]:
# Cell for Correlation Heatmap
corr_matrix = df.drop(columns=['Hour'], errors='ignore').corr()

fig3 = px.imshow(
    corr_matrix,
    title='<b>Feature Correlation Heatmap</b>',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1
)
fig3.update_layout(template='plotly_dark', title_x=0.5, width=800, height=800)
fig3.show(renderer='browser')

In [45]:
# Cell for PCA Box Plots Comparison
fig4 = px.box(
    df, x='Class', y=['V1', 'V2', 'V3', 'V4'],
    title='<b>PCA Features (V1 to V4) Distribution by Transaction Class</b>',
    color='Class'
)
fig4.update_layout(template='plotly_dark', title_x=0.5, xaxis_title='Class (0: Legit, 1: Fraud)', yaxis_title='Feature Value')
fig4.show(renderer='browser')

In [46]:
# Check initial row count
print(f"📊 Rows before dropping duplicates: {df.shape[0]:,}")

# Check how many duplicate rows exist
duplicates_count = df.duplicated().sum()
print(f"🔍 Duplicate rows found: {duplicates_count}")

# Drop duplicates if any exist
if duplicates_count > 0:
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"✨ Rows after dropping duplicates: {df.shape[0]:,}")
else:
    print("✨ No duplicates found. Data is already clean!")

📊 Rows before dropping duplicates: 284,807
🔍 Duplicate rows found: 1081
✨ Rows after dropping duplicates: 283,726


In [47]:
print("📋 Dataset Columns & Data Types:")
print(df.dtypes)

print("\n🎯 Target Variable Distribution Check:")
print(df['Class'].value_counts())
print(f"Fraud Rate: {(df['Class'].sum() / len(df)) * 100:.4f}%")

📋 Dataset Columns & Data Types:
Time      float32
V1        float16
V2        float16
V3        float16
V4        float16
V5        float16
V6        float16
V7        float16
V8        float16
V9        float16
V10       float16
V11       float16
V12       float16
V13       float16
V14       float16
V15       float16
V16       float16
V17       float16
V18       float16
V19       float16
V20       float16
V21       float16
V22       float16
V23       float16
V24       float16
V25       float16
V26       float16
V27       float16
V28       float16
Amount    float32
Class        int8
Hour      float32
dtype: object

🎯 Target Variable Distribution Check:
Class
0    283253
1       473
Name: count, dtype: int64
Fraud Rate: 0.1667%


In [48]:
import numpy as np

# Feature Engineering: Extracting Hour from Time and smoothing Amount
df['Hour'] = (df['Time'] / 3600) % 24
df['Log_Amount'] = np.log1p(df['Amount'])

print("✨ Feature Engineering Completed Successfully!")
display(df[['Time', 'Hour', 'Amount', 'Log_Amount']].head())

✨ Feature Engineering Completed Successfully!


,Time,Hour,Amount,Log_Amount
0,0.0,0.000000,149.619995,5.014760
1,0.0,0.000000,2.690000,1.305627
2,1.0,0.000278,378.660004,5.939276
3,1.0,0.000278,123.500000,4.824306
4,2.0,0.000556,69.989998,4.262539


In [49]:
from sklearn.model_selection import train_test_split

# Separate Features (X) and Target (y)
X = df.drop(columns=['Class'])
y = df['Class']

print(f"📊 Features shape (X): {X.shape}")
print(f"📊 Target shape (y): {y.shape}\n")

# Stratified Split to maintain the 0.1667% fraud ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set rows: {X_train.shape[0]:,}")
print(f"Testing set rows: {X_test.shape[0]:,}")

📊 Features shape (X): (283726, 32)
📊 Target shape (y): (283726,)

Training set rows: 226,980
Testing set rows: 56,746


In [50]:
import sys
!{sys.executable} -m pip install imbalanced-learn


   ---------------------------------------- 0/2 [sklearn-compat]
   ---------------------------------------- 0/2 [sklearn-compat]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imbalanced-learn


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [51]:
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE

# 1. Feature Scaling using RobustScaler (Prevents data leakage by fitting ONLY on X_train)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✨ Feature scaling completed using RobustScaler!")

# 2. Handling Class Imbalance using SMOTE (Applied strictly on training data only)
print("⚖️ Applying SMOTE on Training Data...")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f"Before SMOTE - Training Fraud Cases: {sum(y_train == 1):,}")
print(f"After SMOTE - Training Fraud Cases: {sum(y_train_smote == 1):,}")
print(f"✨ Total Balanced Training Samples: {X_train_smote.shape[0]:,}")

✨ Feature scaling completed using RobustScaler!
⚖️ Applying SMOTE on Training Data...
Before SMOTE - Training Fraud Cases: 378
After SMOTE - Training Fraud Cases: 226,602
✨ Total Balanced Training Samples: 453,204


In [52]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import seaborn as sns
import matplotlib.pyplot as plt

print("🌲 Training Random Forest Classifier on SMOTE-balanced data...")

# 1. Initialize and train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_smote, y_train_smote)

print("✨ Model Training Completed Successfully!\n")

# 2. Predict on the UNTOUCHED Test Data (X_test_scaled)
print("🔍 Evaluating model on Test Data...")
y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

# 3. Print Performance Metrics
print("\n📊 Classification Report (Random Forest):")
print(classification_report(y_test, y_pred_rf, digits=4))

# 4. Calculate ROC-AUC Score
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)
print(f"🌟 ROC-AUC Score: {roc_auc_rf:.4f}")

🌲 Training Random Forest Classifier on SMOTE-balanced data...
✨ Model Training Completed Successfully!

🔍 Evaluating model on Test Data...

📊 Classification Report (Random Forest):
              precision    recall  f1-score   support

           0     0.9996    0.9999    0.9998     56651
           1     0.9136    0.7789    0.8409        95

    accuracy                         0.9995     56746
   macro avg     0.9566    0.8894    0.9203     56746
weighted avg     0.9995    0.9995    0.9995     56746

🌟 ROC-AUC Score: 0.9704


In [53]:
from sklearn.metrics import confusion_matrix
import plotly.figure_factory as ff

# Calculate Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)

# Create interactive heatmap using Plotly
x_labels = ['Predicted Legitimate', 'Predicted Fraud']
y_labels = ['Actual Legitimate', 'Actual Fraud']

fig_cm = ff.create_annotated_heatmap(
    z=cm, x=x_labels, y=y_labels,
    colorscale='Viridis', showscale=True, annotation_text=cm.astype(str)
)

fig_cm.update_layout(
    title='<b>Confusion Matrix - Random Forest (SMOTE Balanced)</b>',
    xaxis_title='Predicted Label',
    yaxis_title='Actual Label',
    template='plotly_dark',
    title_x=0.5
)
fig_cm.show(renderer='browser')

In [54]:
import sys
!{sys.executable} -m pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
    --------------------------------------- 0.8/48.9 MB 2.1 MB/s eta 0:00:23
   - -------------------------------------- 1.3/48.9 MB 2.2 MB/s eta 0:00:22
   - -------------------------------------- 1.6/48.9 MB 2.1 MB/s eta 0:00:23
   - -------------------------------------- 1.8/48.9 MB 2.1 MB/s eta 0:00:23
   - -------------------------------------- 2.4/48.9 MB 1.9 MB/s eta 0:00:25
   -- ------------------------------------- 2.6/48.9 MB 1.9 MB/s eta 0:00:25
   -- ------------------------------------- 3.1/48.9 MB 1.9 MB/s eta 0:00:24
   -- ------------------------------------- 3.4/48.9 MB 1.9 MB/s eta 0:00:24
   --- ------------------------------------ 3.9/48.9 MB 1.9 MB/s eta 0:00:24
   --- ------------------------------------ 4.5/48.9 MB 2.0 MB/s eta 0:00:23
   --- -------------


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [55]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

print("🚀 Training XGBoost Classifier on SMOTE-balanced data...")

# Initialize and train XGBoost
# scale_pos_weight ya SMOTE ke sath XGBoost bohot lethal combination banta hai
xgb_model = XGBClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=6, 
    random_state=42, 
    n_jobs=-1
)

xgb_model.fit(X_train_smote, y_train_smote)
print("✨ XGBoost Training Completed Successfully!\n")

# Predict on Untouched Test Data
print("🔍 Evaluating XGBoost on Test Data...")
y_pred_xgb = xgb_model.predict(X_test_scaled)
y_prob_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]

# Performance Metrics
print("\n📊 Classification Report (XGBoost):")
print(classification_report(y_test, y_pred_xgb, digits=4))

# ROC-AUC Score
roc_auc_xgb = roc_auc_score(y_test, y_prob_xgb)
print(f"🌟 XGBoost ROC-AUC Score: {roc_auc_xgb:.4f}")

🚀 Training XGBoost Classifier on SMOTE-balanced data...
✨ XGBoost Training Completed Successfully!

🔍 Evaluating XGBoost on Test Data...

📊 Classification Report (XGBoost):
              precision    recall  f1-score   support

           0     0.9997    0.9983    0.9990     56651
           1     0.4463    0.8316    0.5809        95

    accuracy                         0.9980     56746
   macro avg     0.7230    0.9149    0.7899     56746
weighted avg     0.9988    0.9980    0.9983     56746

🌟 XGBoost ROC-AUC Score: 0.9786


In [56]:
import numpy as np
from sklearn.metrics import classification_report

# Apne hisab se threshold adjust karke dekho (e.g., 0.8 or 0.85)
custom_threshold = 0.80

# Convert probabilities to binary predictions using the custom threshold
y_pred_custom = (y_prob_xgb >= custom_threshold).astype(int)

print(f"📊 Classification Report at Custom Threshold = {custom_threshold}:")
print(classification_report(y_test, y_pred_custom, digits=4))

📊 Classification Report at Custom Threshold = 0.8:
              precision    recall  f1-score   support

           0     0.9997    0.9996    0.9996     56651
           1     0.7600    0.8000    0.7795        95

    accuracy                         0.9992     56746
   macro avg     0.8798    0.8998    0.8896     56746
weighted avg     0.9993    0.9992    0.9993     56746



In [57]:
import pandas as pd
import plotly.express as px

# Get feature importance from XGBoost model
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False).head(10) # Top 10 features

# Plot interactive horizontal bar chart using Plotly
fig_imp = px.bar(
    importance_df, 
    x='Importance', 
    y='Feature', 
    orientation='h',
    title='<b>Top 10 Feature Importances (XGBoost)</b>',
    color='Importance',
    color_continuous_scale='Viridis'
)
fig_imp.update_layout(
    template='plotly_dark', 
    title_x=0.5, 
    yaxis={'categoryorder':'total ascending'}
)
fig_imp.show(renderer='browser')

In [58]:
import joblib

# Save the trained XGBoost model and RobustScaler for future deployment
joblib.dump(xgb_model, 'credit_card_fraud_xgb_model.pkl')
joblib.dump(scaler, 'robust_scaler.pkl')

print("💾 Model and Scaler successfully saved to disk as 'credit_card_fraud_xgb_model.pkl' and 'robust_scaler.pkl'!")

💾 Model and Scaler successfully saved to disk as 'credit_card_fraud_xgb_model.pkl' and 'robust_scaler.pkl'!


In [59]:
from sklearn.metrics import roc_auc_score

# Training set par probability nikalo (SMOTE balanced data par)
train_prob = xgb_model.predict_proba(X_train_smote)[:, 1]
train_auc = roc_auc_score(y_train_smote, train_prob)

# Test set par probability nikalo
test_prob = xgb_model.predict_proba(X_test_scaled)[:, 1]
test_auc = roc_auc_score(y_test, test_prob)

print(f"🌟 Training ROC-AUC Score: {train_auc:.4f}")
print(f"🌟 Testing ROC-AUC Score:  {test_auc:.4f}")

🌟 Training ROC-AUC Score: 1.0000
🌟 Testing ROC-AUC Score:  0.9786


In [60]:
import os

app_code = '''import streamlit as st
import numpy as np
import pandas as pd
import joblib
import os

# Page Configuration
st.set_page_config(
    page_title="Credit Card Fraud Detection System",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Load Model and Scaler
@st.cache_resource
def load_artifacts():
    model_path = 'credit_card_fraud_xgb_model.pkl'
    scaler_path = 'robust_scaler.pkl'
    
    if not os.path.exists(model_path) or not os.path.exists(scaler_path):
        return None, None
        
    model = joblib.load(model_path)
    scaler = joblib.load(scaler_path)
    return model, scaler

model, scaler = load_artifacts()

# App Header
st.title("🛡️ Credit Card Fraud Detection Dashboard")
st.markdown("Enterprise-grade real-time transaction monitoring powered by **XGBoost**, **RobustScaler**, and **SMOTE**.")

if model is None or scaler is None:
    st.error("⚠️ Model or Scaler files not found! Please ensure 'credit_card_fraud_xgb_model.pkl' and 'robust_scaler.pkl' are in the directory.")
else:
    # Sidebar Configuration
    st.sidebar.header("⚙️ Model Parameters")
    threshold = st.sidebar.slider(
        "Decision Threshold", 
        min_value=0.1, 
        max_value=0.99, 
        value=0.80, 
        step=0.01,
        help="Adjusting threshold higher reduces false positives (false alarms)."
    )
    
    st.sidebar.markdown("---")
    st.sidebar.info(
        "**Pipeline Info:**\\n"
        "- **Model:** XGBoost Classifier\\n"
        "- **Scaling:** RobustScaler\\n"
        "- **Imbalance Handling:** SMOTE\\n"
        "- **Trained ROC-AUC:** 0.9786"
    )

    # Input Form
    st.subheader("💳 Transaction Details Input")
    
    col1, col2 = st.columns(2)
    
    with col1:
        time_val = st.number_input("Transaction Time (seconds from start)", min_value=0.0, max_value=172800.0, value=406.0, step=1.0)
        amount_val = st.number_input("Transaction Amount ($)", min_value=0.0, max_value=50000.0, value=149.62, step=0.01)
    
    with col2:
        st.markdown("**PCA Transformed Features (V1 - V28):**")
        st.caption("Default values represent a typical transaction profile. Adjust as needed.")

    # V features in collapsible expander
    with st.expander("🔬 Advanced PCA Features (V1 to V28)", expanded=False):
        v_cols = {}
        cols = st.columns(4)
        for i in range(1, 29):
            col_idx = (i - 1) % 4
            with cols[col_idx]:
                v_cols[f'V{i}'] = st.number_input(f"V{i}", value=0.0, format="%.4f")

    # Predict Button
    if st.button("🔍 Analyze Transaction for Fraud", type="primary", use_container_width=True):
        # Feature Engineering
        hour_val = (time_val / 3600) % 24
        log_amount_val = np.log1p(amount_val)
        
        # Build feature dictionary matching training columns order
        input_data = {'Time': time_val}
        for i in range(1, 29):
            input_data[f'V{i}'] = v_cols[f'V{i}']
        input_data['Amount'] = amount_val
        input_data['Hour'] = hour_val
        input_data['Log_Amount'] = log_amount_val
        
        input_df = pd.DataFrame([input_data])
        
        # Scale features
        input_scaled = scaler.transform(input_df)
        
        # Predict probability
        fraud_prob = model.predict_proba(input_scaled)[0][1]
        is_fraud = fraud_prob >= threshold
        
        st.markdown("---")
        st.subheader("📊 Analysis Results")
        
        res_col1, res_col2 = st.columns([1, 2])
        
        with res_col1:
            st.metric(label="Fraud Probability", value=f"{fraud_prob*100:.2f}%")
            
        with res_col2:
            if is_fraud:
                st.error(f"🚨 **ALERT: Potential Fraudulent Transaction Detected!** (Confidence: {fraud_prob*100:.2f}%)")
                st.warning("Action Recommended: Flag transaction for immediate manual review or block card temporarily.")
            else:
                st.success(f"✅ **Transaction is Legitimate.** (Confidence: {(1-fraud_prob)*100:.2f}%)")
                st.info("Status: Approved for processing.")
        
        # Probability progress bar
        st.markdown("### Risk Level Meter")
        st.progress(float(fraud_prob))
'''''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print("✨ 'app.py' successfully created in the main folder!")

✨ 'app.py' successfully created in the main folder!
